> **과거 수업·제출 실험 보관본**
> 본문의 수치는 과거 기록이며 현재 재실행 결과가 아닙니다. 개인 고도화의 실행 코드는 `toxicity/`, 검증 집계와 읽는 노트북은 `reports/retrospective-v1.json` 및 `03_reproducible_evaluation.ipynb`를 사용합니다.
> 이 보관본에는 전체 라벨을 참고한 특성 선택, OOF 재사용, 평가 fold 기반 조기 종료가 있어 독립 성능으로 해석하지 않습니다.


# 신약 후보물질 독성 예측 모델링

## 목적

어제 EDA에서 확인한 구조 정보 손실 문제를 실제 예측 성능으로 검증한다.

- **Original**: 기존 ECFP·FCFP·PTFP와 Descriptor
- **Morgan**: 기존 ECFP를 chirality-aware Morgan ECFP로 교체
- **Morgan8**: Morgan에 탐색적 입체 변수 8개 추가
- 같은 5개 fold와 같은 LightGBM 설정으로 비교
- Logistic Regression을 선형 Baseline으로 사용
- OOF 예측으로 F1 임계값 선택
- 최종 제출 파일은 `SMILES`, `output` 두 열로 생성

공식 label은 원본 그대로 사용한다.

- `0 = 독성 있음`
- `1 = 독성 없음`

따라서 제출 전에 label을 다시 반전하지 않는다.

## 실행 환경

노트북과 `train.csv`, `predict_input.csv`, `predict_output_홍길동.csv`를 같은 폴더에 둔다. 처음 실행하는 환경에서는 아래 패키지를 설치한다.

In [ ]:
import sys, subprocess, importlib.util

required = {
    "rdkit": "rdkit",
    "sklearn": "scikit-learn",
    "lightgbm": "lightgbm",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("환경 확인 완료")

## 1. 데이터 로드와 구조 검증

10월 2일 배포된 원본 `train.csv`를 사용한다. 예측 데이터가 학습 데이터에서 label만 빠진 동일 구조인지 검증한다.

In [ ]:
import warnings
from sklearn.model_selection import StratifiedGroupKFold
from rdkit.Chem.Scaffolds import MurckoScaffold
import json
import math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import rdFingerprintGenerator, rdCIPLabeler

ROOT = Path.cwd()
TRAIN_PATH = ROOT / 'train.csv'
PREDICT_PATH = ROOT / 'predict_input.csv'
OUT_DIR = ROOT / 'model_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(TRAIN_PATH)
predict = pd.read_csv(PREDICT_PATH)
assert train.shape == (8349, 3078)
assert predict.shape == (927, 3077)
assert train.drop(columns='label').columns.tolist() == predict.columns.tolist()
assert set(train.label.unique()) == {0, 1}

ecfp_cols = [f'ecfp_{i}' for i in range(1024)]
fcfp_cols = [f'fcfp_{i}' for i in range(1024)]
ptfp_cols = [f'ptfp_{i}' for i in range(1024)]
descriptor_cols = ['MolWt', 'clogp', 'sa_score', 'qed']
all_fp_cols = ecfp_cols + fcfp_cols + ptfp_cols
constant_cols = [c for c in all_fp_cols if train[c].nunique(dropna=False) <= 1]
assert len(constant_cols) == 49
kept_ptfp_cols = [c for c in ptfp_cols if c not in constant_cols]
y = train.label.to_numpy(dtype=np.int8)

## 2. Morgan 및 입체 변수 생성

SMILES에서 `radius=2`, `1,024bit`, `includeChirality=True` Morgan을 생성한다. 입체 변수 8개는 어제 EDA의 탐색 결과를 재현하기 위한 후보 특성이다.

이 8개 중 지역 환경 5개는 전체 학습 label을 참고해 선택됐으므로, 독립적인 예측력은 교차검증 결과로 판단해야 한다.

In [ ]:
def environment(mol, center):
    bonds = list(Chem.FindAtomEnvironmentOfRadiusN(mol, 2, center))
    atoms = {center}
    for bid in bonds:
        bond = mol.GetBondWithIdx(bid)
        atoms.update((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
    return Chem.MolFragmentToSmiles(
        mol, atomsToUse=sorted(atoms), bondsToUse=bonds,
        rootedAtAtom=center, canonical=True, isomericSmiles=False,
    )


LOCAL_KEYS = [
    ('C(C(C)(C)O)(OC)C(c)O', 'R'),
    ('C(C(N)=O)(CN)C(c)C', 'R'),
    ('C(N)(CO)C(O)C', 'R'),
    ('C(O)(C(c)O)C(C)O', 'R'),
    ('C1(CC1C)C(=O)N', 'R'),
]
STEREO_NAMES = [
    'stereo_R_minus_S', 'stereo_r_minus_s', 'stereo_unassigned_count',
    'stereo_env_001', 'stereo_env_006', 'stereo_env_046',
    'stereo_env_054', 'stereo_env_084',
]


def stereo8(mol):
    potential = list(Chem.FindPotentialStereo(mol))
    rdCIPLabeler.AssignCIPLabels(mol)
    centers = {s.centeredOn for s in potential if s.type == Chem.StereoType.Atom_Tetrahedral}
    centers.update(a.GetIdx() for a in mol.GetAtoms() if a.HasProp('_CIPCode'))
    codes = Counter()
    local = Counter()
    for idx in sorted(centers):
        atom = mol.GetAtomWithIdx(idx)
        code = atom.GetProp('_CIPCode') if atom.HasProp('_CIPCode') else '?'
        codes[code] += 1
        local[(environment(mol, idx), code)] += 1
    return np.array([
        codes['R'] - codes['S'],
        codes['r'] - codes['s'],
        codes['?'],
        *[local[key] for key in LOCAL_KEYS],
    ], dtype=np.float32)


generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=1024, includeChirality=True
)


def generate_smiles_features(smiles_values):
    morgan = np.empty((len(smiles_values), 1024), dtype=np.uint8)
    stereo = np.empty((len(smiles_values), 8), dtype=np.float32)
    invalid = []
    for i, smiles in enumerate(smiles_values):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            invalid.append((i, smiles))
            continue
        DataStructs.ConvertToNumpyArray(generator.GetFingerprint(mol), morgan[i])
        stereo[i] = stereo8(mol)
    if invalid:
        raise ValueError(f'Invalid SMILES: {invalid[:5]}')
    return morgan, stereo


train_morgan, train_stereo = generate_smiles_features(train.SMILES)
test_morgan, test_stereo = generate_smiles_features(predict.SMILES)


def csr_from_frame(frame, cols):
    return sparse.csr_matrix(frame[cols].to_numpy(dtype=np.float32))


base_other_train = csr_from_frame(train, fcfp_cols + kept_ptfp_cols)
base_other_test = csr_from_frame(predict, fcfp_cols + kept_ptfp_cols)
descriptor_train = sparse.csr_matrix(train[descriptor_cols].to_numpy(dtype=np.float32))
descriptor_test = sparse.csr_matrix(predict[descriptor_cols].to_numpy(dtype=np.float32))
original_ecfp_train = csr_from_frame(train, ecfp_cols)
original_ecfp_test = csr_from_frame(predict, ecfp_cols)
morgan_train = sparse.csr_matrix(train_morgan.astype(np.float32))
morgan_test = sparse.csr_matrix(test_morgan.astype(np.float32))
stereo_train = sparse.csr_matrix(train_stereo)
stereo_test = sparse.csr_matrix(test_stereo)

feature_sets = {
    'Original': (
        sparse.hstack([original_ecfp_train, base_other_train, descriptor_train], format='csr'),
        sparse.hstack([original_ecfp_test, base_other_test, descriptor_test], format='csr'),
    ),
    'Morgan': (
        sparse.hstack([morgan_train, base_other_train, descriptor_train], format='csr'),
        sparse.hstack([morgan_test, base_other_test, descriptor_test], format='csr'),
    ),
    'Morgan8': (
        sparse.hstack([morgan_train, base_other_train, descriptor_train, stereo_train], format='csr'),
        sparse.hstack([morgan_test, base_other_test, descriptor_test, stereo_test], format='csr'),
    ),
}

folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(np.zeros(len(y)), y))

## 3. 공통 평가 함수

모든 실험에 동일한 `StratifiedKFold(n_splits=5, random_state=42)`를 사용한다. 공식 평가는 label 1의 F1이고, 독성 탐지 성능을 보기 위해 label 0의 F1과 Recall도 함께 기록한다.

In [ ]:
def best_threshold(y_true, prob):
    thresholds = np.arange(0.20, 0.801, 0.005)
    scores = np.array([f1_score(y_true, prob >= t) for t in thresholds])
    idx = int(scores.argmax())
    return float(thresholds[idx]), float(scores[idx])


def metrics(y_true, prob, threshold):
    pred = (prob >= threshold).astype(np.int8)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, pred),
        'precision_label1': precision_score(y_true, pred, pos_label=1),
        'recall_label1': recall_score(y_true, pred, pos_label=1),
        'f1_label1': f1_score(y_true, pred, pos_label=1),
        'recall_label0_toxic': recall_score(y_true, pred, pos_label=0),
        'f1_label0_toxic': f1_score(y_true, pred, pos_label=0),
        'roc_auc': roc_auc_score(y_true, prob),
        'confusion_matrix_0_1': cm.tolist(),
    }

## 4. LightGBM 및 선형 Baseline

고차원 Fingerprint에 적합한 LightGBM을 주 모델로 사용하고 Logistic Regression을 선형 Baseline으로 비교한다. 각 fold의 검증 예측을 모은 OOF 확률에서 공식 F1을 최대화하는 임계값을 선택한다.

In [ ]:
results = []
test_probabilities = {}
oof_probabilities = {}

for feature_name, (X, X_test) in feature_sets.items():
    oof = np.zeros(len(y), dtype=np.float64)
    test_folds = []
    best_iterations = []
    fold_f1 = []
    for fold, (tr_idx, va_idx) in enumerate(folds):
        model = LGBMClassifier(
            objective='binary', n_estimators=1200, learning_rate=0.025,
            num_leaves=31, max_depth=-1, min_child_samples=25,
            subsample=0.85, colsample_bytree=0.55,
            reg_alpha=0.2, reg_lambda=2.0,
            random_state=100 + fold, n_jobs=-1, verbosity=-1,
        )
        model.fit(
            X[tr_idx], y[tr_idx], eval_set=[(X[va_idx], y[va_idx])],
            eval_metric='binary_logloss',
            callbacks=[early_stopping(80, verbose=False), log_evaluation(0)],
        )
        oof[va_idx] = model.predict_proba(X[va_idx])[:, 1]
        test_folds.append(model.predict_proba(X_test)[:, 1])
        best_iterations.append(int(model.best_iteration_ or model.n_estimators_))
        fold_f1.append(float(f1_score(y[va_idx], oof[va_idx] >= 0.5)))
    test_prob = np.mean(test_folds, axis=0)
    threshold, _ = best_threshold(y, oof)
    row = {
        'model': 'LightGBM', 'features': feature_name,
        'rows': X.shape[0], 'columns': X.shape[1],
        'cv_f1_at_0.5_mean': float(np.mean(fold_f1)),
        'cv_f1_at_0.5_std': float(np.std(fold_f1)),
        'best_iterations_mean': float(np.mean(best_iterations)),
        **metrics(y, oof, threshold),
    }
    results.append(row)
    test_probabilities[f'LGBM_{feature_name}'] = test_prob
    oof_probabilities[f'LGBM_{feature_name}'] = oof

# Interpretable linear baseline on Original and Morgan8.
for feature_name in ['Original', 'Morgan8']:
    X, X_test = feature_sets[feature_name]
    oof = np.zeros(len(y), dtype=np.float64)
    test_folds = []
    fold_f1 = []
    for fold, (tr_idx, va_idx) in enumerate(folds):
        model = make_pipeline(
            MaxAbsScaler(),
            LogisticRegression(
                C=0.25, solver='liblinear', penalty='l2',
                max_iter=1000, random_state=200 + fold,
            ),
        )
        model.fit(X[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict_proba(X[va_idx])[:, 1]
        test_folds.append(model.predict_proba(X_test)[:, 1])
        fold_f1.append(float(f1_score(y[va_idx], oof[va_idx] >= 0.5)))
    test_prob = np.mean(test_folds, axis=0)
    threshold, _ = best_threshold(y, oof)
    results.append({
        'model': 'LogisticRegression', 'features': feature_name,
        'rows': X.shape[0], 'columns': X.shape[1],
        'cv_f1_at_0.5_mean': float(np.mean(fold_f1)),
        'cv_f1_at_0.5_std': float(np.std(fold_f1)),
        'best_iterations_mean': None,
        **metrics(y, oof, threshold),
    })
    test_probabilities[f'LR_{feature_name}'] = test_prob
    oof_probabilities[f'LR_{feature_name}'] = oof

## 5. 확률 Ensemble

In [ ]:
# Candidate ensemble: average the three strongest nonlinear feature views.
ensemble_oof = np.mean([
    oof_probabilities['LGBM_Original'],
    oof_probabilities['LGBM_Morgan'],
    oof_probabilities['LGBM_Morgan8'],
], axis=0)
ensemble_test = np.mean([
    test_probabilities['LGBM_Original'],
    test_probabilities['LGBM_Morgan'],
    test_probabilities['LGBM_Morgan8'],
], axis=0)
ensemble_threshold, _ = best_threshold(y, ensemble_oof)
results.append({
    'model': 'ProbabilityEnsemble', 'features': 'Original+Morgan+Morgan8',
    'rows': len(y), 'columns': None,
    'cv_f1_at_0.5_mean': None, 'cv_f1_at_0.5_std': None,
    'best_iterations_mean': None,
    **metrics(y, ensemble_oof, ensemble_threshold),
})
test_probabilities['Ensemble'] = ensemble_test
oof_probabilities['Ensemble'] = ensemble_oof

results_df = pd.DataFrame(results).sort_values('f1_label1', ascending=False)
results_df.to_csv(OUT_DIR / 'cv_results.csv', index=False)

## 실제 교차검증 결과

| 모델 | 특성 | 공식 F1(label 1) | 독성 F1(label 0) | 독성 Recall | ROC-AUC |
|---|---|---:|---:|---:|---:|
| Probability Ensemble | Original+Morgan+Morgan8 | **0.8268** | **0.7763** | **0.7428** | **0.8875** |
| LightGBM | Original | 0.8267 | 0.7698 | 0.7250 | 0.8864 |
| LightGBM | Morgan8 | 0.8265 | 0.7658 | 0.7147 | 0.8862 |
| LightGBM | Morgan | 0.8260 | 0.7330 | 0.6344 | 0.8867 |
| Logistic Regression | Original | 0.7993 | 0.7251 | 0.6711 | 0.8452 |
| Logistic Regression | Morgan8 | 0.7991 | 0.7190 | 0.6572 | 0.8447 |

### 해석

- Ensemble이 가장 높지만 Original LightGBM과 차이는 0.0001 수준으로 사실상 동률이다.
- Morgan은 기존 숫자 입력 충돌을 줄였지만 예측 F1을 개선하지 않았다.
- Morgan+8도 충돌 0건을 만들었지만 독립 예측 성능은 개선하지 못했다.
- 충돌 제거와 일반화 성능은 다른 문제라는 점이 확인됐다.
- 가장 안정적인 단일 모델은 기존 특성 LightGBM이다.

## 6. 제출 후보 생성

예측 입력의 SMILES 순서를 유지하고 `output`에 원본 label 규칙의 예측값을 기록한다. 최대 4회 제출할 수 있도록 특성 조합이 다른 후보를 생성한다.

In [ ]:
# Four leaderboard candidates, preserving template columns and SMILES order.
candidate_specs = []
ranked_lgbm = results_df[results_df.model == 'LightGBM']
for _, row in ranked_lgbm.head(3).iterrows():
    key = f"LGBM_{row['features']}"
    candidate_specs.append((key, float(row['threshold'])))
candidate_specs.append(('Ensemble', float(results_df.loc[results_df.model == 'ProbabilityEnsemble', 'threshold'].iloc[0])))

template = pd.read_csv(ROOT / 'predict_output_홍길동.csv')
assert template.SMILES.equals(predict.SMILES)
submission_manifest = []
for rank, (key, threshold) in enumerate(candidate_specs, 1):
    submission = pd.DataFrame({
        'SMILES': predict.SMILES,
        'output': (test_probabilities[key] >= threshold).astype(np.int8),
    })
    path = OUT_DIR / f'candidate_{rank}_{key}.csv'
    submission.to_csv(path, index=False)
    submission_manifest.append({
        'rank': rank, 'candidate': key, 'threshold': threshold,
        'predicted_0': int((submission.output == 0).sum()),
        'predicted_1': int((submission.output == 1).sum()),
        'path': str(path),
    })

audit = {
    'python_label_definition': {'0': 'toxic', '1': 'non-toxic'},
    'submission_label_definition': {'0': 'toxic', '1': 'non-toxic'},
    'rdkit_version': rdBase.rdkitVersion,
    'constant_columns_removed': constant_cols,
    'stereo_features': STEREO_NAMES,
    'feature_shapes': {k: [int(v[0].shape[0]), int(v[0].shape[1])] for k, v in feature_sets.items()},
    'results': results,
    'submissions': submission_manifest,
}
(OUT_DIR / 'model_audit.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2))

print(results_df[[
    'model', 'features', 'columns', 'threshold', 'f1_label1',
    'f1_label0_toxic', 'recall_label0_toxic', 'roc_auc',
]].to_string(index=False))
print('\nSUBMISSIONS')
print(pd.DataFrame(submission_manifest).to_string(index=False))

## 제출 추천 순서

1. `Ensemble`: 교차검증 공식 F1과 독성 Recall이 모두 가장 높음
2. `LGBM_Original`: 가장 안정적인 단일 모델
3. `LGBM_Morgan8`: 어제 EDA 개선안의 실제 제출 성능 확인
4. `LGBM_Morgan`: chirality-aware Morgan 단독 교체 효과 확인

Leaderboard 결과는 교차검증과 다를 수 있다. 첫 제출 결과를 확인한 후 남은 제출 기회를 사용한다.

# 논리 감사: 무작위 분할의 한계 검증

## 발견한 잠재적 허점

1. 무작위 Stratified 분할에서는 유사한 분자 골격이 학습과 검증에 함께 들어갈 수 있다.
2. 전체 OOF에서 임계값을 선택하고 같은 OOF로 점수를 계산하면 임계값 선택 효과가 낙관적으로 포함된다.
3. 입체 변수 8개 중 지역 환경 5개는 전체 label을 참고해 선택됐다.
4. 기존 LightGBM 설정은 `subsample_freq=0`이라 `subsample=0.85`가 실제 적용되지 않았다.

아래 검증에서는 chirality를 제거한 Bemis-Murcko scaffold를 그룹으로 사용한다. 고리가 없는 분자는 chirality-free canonical connectivity로 구분한다. 같은 그룹은 서로 다른 fold에 들어가지 않는다.

임계값도 각 검증 fold를 제외한 나머지 네 fold의 OOF 예측에서 선택하는 cross-fitted 방식으로 평가한다.

In [ ]:
def scaffold_group(smiles):
    mol = Chem.MolFromSmiles(smiles)
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    # Acyclic molecules have an empty Murcko scaffold. Avoid putting all of them
    # into one giant group; use chirality-free canonical connectivity instead.
    if not scaffold:
        scaffold = 'ACYCLIC:' + Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
    return scaffold


groups = np.array([scaffold_group(s) for s in train.SMILES], dtype=object)
print('rows', len(y), 'unique_scaffold_groups', len(np.unique(groups)), 'largest_group', pd.Series(groups).value_counts().iloc[0])

folds = list(
    StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    .split(np.zeros(len(y)), y, groups)
)

for fold, (tr, va) in enumerate(folds):
    overlap = set(groups[tr]) & set(groups[va])
    assert not overlap
    print('fold', fold, 'train', len(tr), 'valid', len(va), 'valid_positive_rate', round(float(y[va].mean()), 4))


def choose_threshold(y_true, prob):
    thresholds = np.arange(0.20, 0.801, 0.005)
    scores = [f1_score(y_true, prob >= t, pos_label=1) for t in thresholds]
    return float(thresholds[int(np.argmax(scores))])


results = []
oof_by_feature = {}
test_by_feature = {}

for feature_name, (X, X_test) in feature_sets.items():
    oof = np.zeros(len(y), dtype=float)
    fold_ids = np.empty(len(y), dtype=np.int8)
    iterations = []
    test_fold_prob = []
    for fold, (tr, va) in enumerate(folds):
        model = LGBMClassifier(
            objective='binary', n_estimators=1200, learning_rate=0.025,
            num_leaves=31, max_depth=-1, min_child_samples=25,
            subsample=0.85, subsample_freq=1, colsample_bytree=0.55,
            reg_alpha=0.2, reg_lambda=2.0,
            random_state=300 + fold, n_jobs=-1, verbosity=-1,
        )
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            model.fit(
                X[tr], y[tr], eval_set=[(X[va], y[va])],
                eval_metric='binary_logloss',
                callbacks=[early_stopping(80, verbose=False), log_evaluation(0)],
            )
        oof[va] = model.predict_proba(X[va])[:, 1]
        test_fold_prob.append(model.predict_proba(X_test)[:, 1])
        fold_ids[va] = fold
        iterations.append(int(model.best_iteration_ or model.n_estimators_))

    # Fixed 0.5 evaluation is fully independent of validation labels.
    pred_fixed = oof >= 0.5

    # Cross-fitted threshold: each validation fold's threshold is selected only
    # from the other four folds' OOF predictions.
    pred_crossfit = np.zeros(len(y), dtype=bool)
    thresholds = []
    for fold in range(5):
        threshold = choose_threshold(y[fold_ids != fold], oof[fold_ids != fold])
        thresholds.append(threshold)
        pred_crossfit[fold_ids == fold] = oof[fold_ids == fold] >= threshold

    row = {
        'features': feature_name,
        'fixed_f1_label1': f1_score(y, pred_fixed, pos_label=1),
        'crossfit_f1_label1': f1_score(y, pred_crossfit, pos_label=1),
        'crossfit_f1_label0_toxic': f1_score(y, pred_crossfit, pos_label=0),
        'crossfit_recall_label0_toxic': recall_score(y, pred_crossfit, pos_label=0),
        'roc_auc': roc_auc_score(y, oof),
        'thresholds': thresholds,
        'best_iterations_mean': float(np.mean(iterations)),
    }
    results.append(row)
    oof_by_feature[feature_name] = oof
    test_by_feature[feature_name] = np.mean(test_fold_prob, axis=0)

# Honest ensemble threshold evaluation using cross-fitted thresholds.
ensemble = np.mean([oof_by_feature[k] for k in ['Original', 'Morgan', 'Morgan8']], axis=0)
ensemble_test = np.mean([test_by_feature[k] for k in ['Original', 'Morgan', 'Morgan8']], axis=0)
pred_fixed = ensemble >= 0.5
pred_crossfit = np.zeros(len(y), dtype=bool)
thresholds = []
for fold in range(5):
    threshold = choose_threshold(y[fold_ids != fold], ensemble[fold_ids != fold])
    thresholds.append(threshold)
    pred_crossfit[fold_ids == fold] = ensemble[fold_ids == fold] >= threshold
results.append({
    'features': 'Ensemble',
    'fixed_f1_label1': f1_score(y, pred_fixed, pos_label=1),
    'crossfit_f1_label1': f1_score(y, pred_crossfit, pos_label=1),
    'crossfit_f1_label0_toxic': f1_score(y, pred_crossfit, pos_label=0),
    'crossfit_recall_label0_toxic': recall_score(y, pred_crossfit, pos_label=0),
    'roc_auc': roc_auc_score(y, ensemble),
    'thresholds': thresholds,
    'best_iterations_mean': None,
})

result = pd.DataFrame(results).sort_values('crossfit_f1_label1', ascending=False)
print(result.to_string(index=False))
result.to_json(OUT_DIR / 'logical_audit_results.json', orient='records', force_ascii=False, indent=2)

# Use the median of the five independently selected thresholds for test data.
threshold_map = {
    row['features']: float(np.median(row['thresholds']))
    for row in results
}
test_by_feature['Ensemble'] = ensemble_test
submission_order = ['Original', 'Ensemble', 'Morgan', 'Morgan8']
for rank, name in enumerate(submission_order, 1):
    threshold = threshold_map[name]
    submission = pd.DataFrame({
        'SMILES': predict.SMILES,
        'output': (test_by_feature[name] >= threshold).astype(np.int8),
    })
    path = OUT_DIR / f'scaffold_candidate_{rank}_{name}.csv'
    submission.to_csv(path, index=False)
    print('submission', rank, name, 'threshold', threshold, submission.output.value_counts().sort_index().to_dict())

## 엄격한 Scaffold Group 교차검증 결과

| 순위 | 특성 | 공식 F1 | 독성 F1 | 독성 Recall | ROC-AUC |
|---:|---|---:|---:|---:|---:|
| 1 | Original | **0.8026** | 0.7232 | 0.6601 | 0.8514 |
| 2 | Ensemble | 0.8016 | 0.7193 | 0.6533 | **0.8525** |
| 3 | Morgan8 | 0.8002 | 0.7147 | 0.6454 | 0.8514 |
| 4 | Morgan | 0.8000 | **0.7312** | **0.6840** | 0.8507 |

### 최종 해석

- 무작위 분할의 최고 F1 0.8268은 골격 분할에서 0.8016으로 하락했다. 구조적으로 유사한 화합물이 fold를 넘나들며 점수를 높였을 가능성이 확인됐다.
- 공식 F1 기준으로는 Original LightGBM이 가장 안정적이다.
- Morgan은 공식 F1은 조금 낮지만 독성 Recall과 독성 F1이 가장 높아 안전성 관점의 보조 후보 가치가 있다.
- Morgan8은 충돌 0건을 만들었지만 일반화 성능을 개선하지 못했다. label을 참고해 선택한 8개 변수를 최종 개선으로 주장하면 안 된다.
- 제출 우선순위는 Original → Ensemble → Morgan → Morgan8이다.

이 결과를 최종 판단의 기준으로 사용하고, 앞의 무작위 분할 결과는 비교용으로만 유지한다.

## 2차 실험: 다른 모델과 임계값 조정

1차 Leaderboard 결과는 F1 0.8360, Precision 0.7847, Recall 0.8945였다. 정수 혼동행렬은 `TN=313, FP=121, FN=52, TP=441`로 유일하게 복원된다. Recall이 Precision보다 높으므로 다음 가설은 **양성(1) 예측 일부를 보수적으로 줄이면 FP 감소 폭이 TP 감소 폭보다 커져 F1이 개선될 수 있다**는 것이다.

동일한 chirality-free Bemis-Murcko Scaffold 5-fold에서 LightGBM, 보수적 LightGBM, ExtraTrees, Logistic Regression과 확률 앙상블을 비교했다. 임계값은 각 검증 fold 밖의 OOF 예측으로만 선택했다.

| 모델 | Cross-fit F1 | Precision | Recall | AUC |
|---|---:|---:|---:|---:|
| LightGBM + ExtraTrees (50:50) | **0.8037** | **0.7597** | 0.8531 | **0.8521** |
| LightGBM + ExtraTrees (75:25) | 0.8035 | 0.7538 | 0.8602 | 0.8540 |
| LightGBM 기준 | 0.8000 | 0.7412 | **0.8690** | 0.8517 |
| 보수적 LightGBM | 0.7963 | 0.7412 | 0.8602 | 0.8496 |
| ExtraTrees | 0.7918 | 0.7361 | 0.8567 | 0.8328 |
| Logistic Regression | 0.7688 | 0.7028 | 0.8485 | 0.8062 |

해석:

- 다른 모델을 시험한 가치는 있었지만 ExtraTrees와 Logistic Regression 단독은 채택하지 않는다.
- ExtraTrees는 LightGBM과 오류가 달라 확률 앙상블에서만 소폭 개선됐다.
- 1차 제출 대비 threshold 0.475 후보는 정확히 27개 행을 `1→0`으로 바꾸고 역방향 변화는 없다.
- 이 27개 중 실제 양성이 11개 이하라면 1차 F1보다 개선된다. 따라서 2차 제출로 검증할 수 있는 명확한 가설이다.
- Leaderboard를 반복해서 미세 조정하지 않기 위해 threshold 후보는 0.450과 0.475 두 개만 사전에 정했다.


In [ ]:
# 재현용 핵심 설정
# 전체 실행 코드는 별도 실험 스크립트에서 수행했으며, 아래가 비교 설계의 핵심이다.
from sklearn.ensemble import ExtraTreesClassifier

extra_trees = ExtraTreesClassifier(
    n_estimators=500, min_samples_leaf=2, max_features=0.35,
    random_state=700, n_jobs=-1
)
# 각 scaffold fold에서 predict_proba를 생성한 뒤:
# blend_probability = 0.5 * lgbm_probability + 0.5 * extra_trees_probability
# threshold는 해당 fold를 제외한 나머지 OOF에서 선택한다.
